In [1]:
import json
import pandas as pd
import numpy as np
from datetime import datetime
import prettytable as pt
from matplotlib import pyplot as plt
import os 
import warnings
warnings.filterwarnings("ignore")

###################################################################################################
                                                                                                  #
from rhoova.Client import *                                                                       #   
#Register and get api key from https://app.rhoova.com/ for ClientConfig("api key", "api secret")  #
config = ClientConfig("", "")                                                                     #
api = Api(config)                                                                                 # 
                                                                                                  #
###################################################################################################

directory = os.path.normpath(os.getcwd() + os.sep + os.pardir)
datadirectory=directory+"/data/yielddata/yielddata.csv"

yielddata = pd.read_csv(datadirectory)
yielddata = yielddata.replace(np.nan, '', regex=True) 


In [2]:
# Fix parameters for building yield curve instruments
instrumentsconfig={
            "DEPO": {
                "businessDayConvention": "ModifiedFollowing",
                "dayCounter": "Actual360"
            },
            "FUTURES": {
                "businessDayConvention": "ModifiedFollowing",
                "dayCounter": "Actual360",
                "market": "IMM",
                "usedates": True
            },
            "IRS": {
                "businessDayConvention": "ModifiedFollowing",
                "dayCounter": "Actual360",
                "frequency": "Semiannual"
            }
        } 

In [3]:
# Fix parameters for building yield curve

yieldcurveconfig={
        "settlementDays": 2,
        "intpMethod": "LINEAR",
        "currency": "USD",
        "calendar": "UnitedStates",
        "dayCounter": "Actual360",
        "period": "3M",
        "instruments":instrumentsconfig,}

In [8]:
#get specific date zero rates
yieldcurve =  {
    "valuationDate": "2021-06-02",
    "zeroRates": {
        "analysisDates":['2021-12-06','2022-06-06','2022-12-05','2023-06-05','2023-12-04',
                        '2024-06-04','2024-12-04','2025-06-04','2025-12-04','2026-06-04'],
        "compounding": "Continuous",
        "frequency": "Annual"
    },
    "yieldCurve":yieldcurveconfig ,
    "yieldData": yielddata.to_dict('r')
}
try:
    result = api.createTask(CalculationType.YIELD_CURVE, yieldcurve,True)
    result=json.loads(result["result"])
except RhoovaError as e:
    e.printPretty()

In [30]:
zerorates=result.get("zeroRates"),
discountrates=result.get("discountRates")
dates=[]
forwarddates= {}
rhoovayieldcurvedicts = {}
rhoovayieldcurvediscounts = {}
rhoovaforwardrates={}
for rhvzr in zerorates[0]:
    rhoovayieldcurvedicts[datetime.strptime(rhvzr.get('date'),'%Y-%m-%d').date()] = rhvzr.get('rate')*100
    dates.append(rhvzr.get('date'))


In [31]:
pTable = pt.PrettyTable(['Maturity','Rhoova ZeroRates'])
i=0
for key, val in rhoovayieldcurvedicts.items():
    pTable.add_row([key, val])
    i=i+1
pTable.align = 'c'
pTable.float_format = '.6'
print(pTable)

+------------+------------------+
|  Maturity  | Rhoova ZeroRates |
+------------+------------------+
| 2021-06-04 |     0.128478     |
| 2021-09-07 |     0.128478     |
| 2021-09-16 |     0.126489     |
| 2021-12-15 |     0.124726     |
| 2022-03-15 |     0.139854     |
| 2022-06-16 |     0.144188     |
| 2022-09-15 |     0.153638     |
| 2022-12-21 |     0.172117     |
| 2023-06-05 |     0.227698     |
| 2024-06-04 |     0.421333     |
| 2025-06-04 |     0.660654     |
| 2026-06-04 |     0.882459     |
| 2027-06-04 |     1.083409     |
| 2028-06-05 |     1.254609     |
| 2029-06-04 |     1.389405     |
| 2030-06-04 |     1.499998     |
| 2031-06-04 |     1.593841     |
| 2032-06-04 |     1.674199     |
| 2033-06-06 |     1.744227     |
| 2036-06-04 |     1.888316     |
| 2041-06-04 |     2.007056     |
| 2046-06-04 |     2.043077     |
| 2051-06-05 |     2.047829     |
| 2061-06-06 |     1.944958     |
| 2071-06-04 |     1.812822     |
+------------+------------------+


In [32]:
#get specific date zero rates
yieldcurve =  {
    "valuationDate": "2021-06-02",
    "discountRates": {
        "analysisDates":['2021-12-06','2022-06-06','2022-12-05','2023-06-05','2023-12-04',
                        '2024-06-04','2024-12-04','2025-06-04','2025-12-04','2026-06-04']    },
    "yieldCurve":yieldcurveconfig ,
    "yieldData": yielddata.to_dict('r')
}
try:
    result = api.createTask(CalculationType.YIELD_CURVE, yieldcurve,True)
    result=json.loads(result["result"])
except RhoovaError as e:
    e.printPretty()

In [33]:
discountrates=result.get("discountRates")
for rhv in discountrates:
    rhoovayieldcurvediscounts[datetime.strptime(rhv.get('date'), '%Y-%m-%d').date()] = rhv.get('rate')

In [34]:
pTable = pt.PrettyTable(['Maturity','Rhoova Discount Rates'])
i=0
for key, val in rhoovayieldcurvediscounts.items():
    pTable.add_row([key, val])
    i=i+1
pTable.align = 'c'
pTable.float_format = '.6'
print(pTable)

+------------+-----------------------+
|  Maturity  | Rhoova Discount Rates |
+------------+-----------------------+
| 2021-12-06 |        0.999358       |
| 2022-06-06 |        0.998536       |
| 2022-12-05 |        0.997425       |
| 2023-06-05 |        0.995387       |
| 2023-12-04 |        0.991810       |
| 2024-06-04 |        0.987255       |
| 2024-12-04 |        0.980952       |
| 2025-06-04 |        0.973545       |
| 2025-12-04 |        0.965366       |
| 2026-06-04 |        0.956227       |
+------------+-----------------------+


In [40]:
#get specific date zero rates
yieldcurve =  {
    "valuationDate": "2021-06-02",
     "forwardRates": {
        "startEndDates": [
            {
                "startDate": "2021-12-06",
                "endDate": "2022-03-04"
            },
            {
                "startDate": "2022-03-04",
                "endDate": "2022-06-06"
            },
            {
                "startDate": "2022-06-06",
                "endDate": "2022-09-06"
            },
            {
                "startDate": "2022-12-05",
                "endDate": "2023-03-06"
            }
        ],
        "compounding": "Continuous",
        "frequency": "Annual"
    },
    "yieldCurve":yieldcurveconfig ,
    "yieldData": yielddata.to_dict('r')
}
try:
    result = api.createTask(CalculationType.YIELD_CURVE, yieldcurve,True)
    result=json.loads(result["result"])
except RhoovaError as e:
    e.printPretty()

In [41]:
forwardrates=result.get("forwardRates")
for rhv in forwardrates:
    rhoovaforwardrates[datetime.strptime(rhv.get('date'), '%Y-%m-%d').date()] = rhv.get('rate')

In [42]:
pTable = pt.PrettyTable(['StartDate','Rhoova Forward Rates'])
i=0
for key, val in rhoovaforwardrates.items():
    pTable.add_row([key, val])
    i=i+1
pTable.align = 'c'
pTable.float_format = '.6'
print(pTable)

+------------+----------------------+
| StartDate  | Rhoova Forward Rates |
+------------+----------------------+
| 2021-12-06 |       0.001656       |
| 2022-03-04 |       0.001603       |
| 2022-06-06 |       0.001885       |
| 2022-12-05 |       0.003671       |
+------------+----------------------+
